In [4]:
import math

def demo(name, before, after):
    print(f"{name:<16} {before}  ->  {after}")

## 1. Bubble Sort
Repeatedly walk the list and swap adjacent elements that are out of order. After pass `i`,the largest `i+1` elements have "bubbled" to the end, so the inner loop can shrink each time.
- **Time:** O(n²) but O(n) best case only if you add an early-exit flag
- **Space:** O(1), in-place, stable

**Use when:** the list is tiny or nearly sorted. Otherwise almost never.

In [5]:
def bubbleSort(customList):
    for i in range(len(customList) - 1):
        for j in range(len(customList) - i - 1):
            if customList[j] > customList[j + 1]:
                customList[j], customList[j + 1] = customList[j + 1], customList[j]
    return customList


# Example
nums = [2, 1, 7, 6, 5, 3, 4, 9, 8]
demo("bubbleSort", list(nums), bubbleSort(nums))

# Edge cases
print(bubbleSort([]))
print(bubbleSort([1]))
print(bubbleSort([5, 4, 3, 2, 1]))

bubbleSort       [2, 1, 7, 6, 5, 3, 4, 9, 8]  ->  [1, 2, 3, 4, 5, 6, 7, 8, 9]
[]
[1]
[1, 2, 3, 4, 5]


## 2. Selection Sort
Find the minimum of the unsorted portion and swap it into position `i`. Repeat.Unlike bubble sort it performs at most **n swaps**, which matters when writes are expensive.
- **Time:** O(n²) in every case
it never gets to exit early
- **Space:** O(1),

in-place, **not stable** (the long-range swap can reorder equal keys)

In [6]:
def selectionSort(customList):
    for i in range(len(customList)):
        min_index = i
        for j in range(i + 1, len(customList)):
            if customList[min_index] > customList[j]:
                min_index = j
        customList[i], customList[min_index] = customList[min_index], customList[i]
    return customList


# Example
nums = [2, 1, 7, 6, 5, 3, 4, 9, 8]
demo("selectionSort", list(nums), selectionSort(nums))

print(selectionSort([64, 25, 12, 22, 11]))

selectionSort    [2, 1, 7, 6, 5, 3, 4, 9, 8]  ->  [1, 2, 3, 4, 5, 6, 7, 8, 9]
[11, 12, 22, 25, 64]


## 3. Insertion Sort
Treat the left side as sorted. Take the next element (`key`), shift everything largerone slot right, and drop `key` into the gap — the way you sort a hand of playing cards.
- **Time:** O(n²) worst, **O(n) if already sorted** (the `while` never runs) 
- **Space:** O(1), in-place, stable
- **Use when:** n is small or data is nearly sorted. This is why Timsort falls back to it  for short runs, and why `bucketSort` below uses it per bucket.

In [7]:
def insertionSort(customList):
    for i in range(1, len(customList)):
        key = customList[i]
        j = i - 1
        while j >= 0 and key < customList[j]:
            customList[j + 1] = customList[j]
            j -= 1
        customList[j + 1] = key
    return customList


# Example
nums = [2, 1, 7, 6, 5, 3, 4, 9, 8]
demo("insertionSort", list(nums), insertionSort(nums))

# Best case: already sorted, inner while loop never executes
print(insertionSort([1, 2, 3, 4, 5]))

insertionSort    [2, 1, 7, 6, 5, 3, 4, 9, 8]  ->  [1, 2, 3, 4, 5, 6, 7, 8, 9]
[1, 2, 3, 4, 5]


## 4. Bucket Sort
Scatter values into `√n` buckets by magnitude, sort each bucket (with insertion sort),then concatenate the buckets in order.
- **Time:** O(n + k) average when values are spread evenly; **O(n²)** if everything lands  in one bucket
- **Space:** O(n)
- not in-place
- **Caveat:** this implementation indexes buckets with `j * numberofBuckets / maxValue`,  so it assumes **positive numbers**. A `0` or a negative value produces index `-1` and  silently lands in the wrong bucket.

In [8]:
def bucketSort(customList):
    numberofBuckets = round(math.sqrt(len(customList)))
    maxValue = max(customList)
    arr = []

    for i in range(numberofBuckets):
        arr.append([])

    # Scatter: place each value into the bucket matching its magnitude
    for j in customList:
        index_b = math.ceil(j * numberofBuckets / maxValue)
        arr[index_b - 1].append(j)

    # Sort each bucket individually
    for i in range(numberofBuckets):
        arr[i] = insertionSort(arr[i])

    # Gather: concatenate buckets back in order
    k = 0
    for i in range(numberofBuckets):
        for j in range(len(arr[i])):
            customList[k] = arr[i][j]
            k += 1
    return customList


# Example (positive values only)
nums = [2, 1, 7, 6, 5, 3, 4, 9, 8]
demo("bucketSort", list(nums), bucketSort(nums))

print(bucketSort([22, 45, 12, 8, 10, 6, 72, 81, 33, 18, 50, 14]))

bucketSort       [2, 1, 7, 6, 5, 3, 4, 9, 8]  ->  [1, 2, 3, 4, 5, 6, 7, 8, 9]
[6, 8, 10, 12, 14, 18, 22, 33, 45, 50, 72, 81]


## 5. Merge Sort
Divide the list in half, sort each half recursively, then **merge** the two sorted halvesby repeatedly taking the smaller front element.`merge()` copies both halves into temporary lists `L` and `R`, then writes back into`customList[l..r]` in sorted order.
- **Time:** O(n log n) in **all** cases — no bad input
- **Space:** O(n) for the temporary arrays
- **Stable**, which is why it's the go-to when equal keys must keep their relative order

In [9]:
def merge(customList, l, m, r):
    n1 = m - l + 1
    n2 = r - m

    L = [0] * n1
    R = [0] * n2

    for i in range(0, n1):
        L[i] = customList[l + i]
    for j in range(0, n2):
        R[j] = customList[m + 1 + j]

    i = 0
    j = 0
    k = l
    # Take the smaller front element from L or R
    while i < n1 and j < n2:
        if L[i] <= R[j]:
            customList[k] = L[i]
            i += 1
        else:
            customList[k] = R[j]
            j += 1
        k += 1

    # Drain whatever is left over
    while i < n1:
        customList[k] = L[i]
        i += 1
        k += 1
    while j < n2:
        customList[k] = R[j]
        j += 1
        k += 1


def mergeSort(customList, l, r):
    if l < r:
        m = (l + (r - 1)) // 2
        mergeSort(customList, l, m)
        mergeSort(customList, m + 1, r)
        merge(customList, l, m, r)
    return customList


# Example -- note the explicit low/high bounds
nums = [2, 1, 7, 6, 5, 3, 4, 9, 8]
demo("mergeSort", list(nums), mergeSort(nums, 0, len(nums) - 1))

nums2 = [38, 27, 43, 3, 9, 82, 10]
print(mergeSort(nums2, 0, len(nums2) - 1))

mergeSort        [2, 1, 7, 6, 5, 3, 4, 9, 8]  ->  [1, 2, 3, 4, 5, 6, 7, 8, 9]
[3, 9, 10, 27, 38, 43, 82]


## 6. Quick Sort
Pick a **pivot** (here: the last element). `partition()` rearranges the range so everything`<= pivot` sits left of it and everything larger sits right, then returns the pivot's finalindex. Recurse on both sides.
- **Time:** O(n log n) average, **O(n²) worst** (already-sorted input with a last-element  pivot gives maximally unbalanced splits)
- **Space:** O(log n) recursion stack 
- in-place, no auxiliary array, **Not stable**
- `quickSort` sorts **in place** and returns `None`, so print the list itself afterwards

In [10]:
def partition(customList, low, high):
    i = low - 1
    pivot = customList[high]
    for j in range(low, high):
        if customList[j] <= pivot:
            i += 1
            customList[i], customList[j] = customList[j], customList[i]
    customList[i + 1], customList[high] = customList[high], customList[i + 1]
    return i + 1


def quickSort(customList, low, high):
    if low < high:
        pi = partition(customList, low, high)
        quickSort(customList, low, pi - 1)
        quickSort(customList, pi + 1, high)


# Example -- sorts in place, returns None
nums = [2, 1, 7, 6, 5, 3, 4, 9, 8]
before = list(nums)
quickSort(nums, 0, len(nums) - 1)
demo("quickSort", before, nums)

nums2 = [10, 80, 30, 90, 40, 50, 70]
quickSort(nums2, 0, len(nums2) - 1)
print(nums2)

quickSort        [2, 1, 7, 6, 5, 3, 4, 9, 8]  ->  [1, 2, 3, 4, 5, 6, 7, 8, 9]
[10, 30, 40, 50, 70, 80, 90]


## 7. Heap Sort
Build a heap from the list, then repeatedly swap the root to the end and re-heapify theshrinking prefix.⚠️ **Watch the comparison direction.** `heapify` below uses `<`, which builds a **min-heap**,so the smallest element is repeatedly moved to the back — the result comes out in**descending order**. The original file had a `customList.reverse()` line commented out forexactly this reason. Flip both comparisons to `>` (a max-heap) to get ascending orderdirectly; both versions are shown.
- **Time:** O(n log n) in all cases
- **Space:** O(1) 
- in-place, and unlike merge sort it needs no extra array
- **Not stable**

In [11]:
def heapify(customList, n, i):
    smallest = i
    l = 2 * i + 1
    r = 2 * i + 2
    if l < n and customList[l] < customList[smallest]:
        smallest = l
    if r < n and customList[r] < customList[smallest]:
        smallest = r
    if smallest != i:
        customList[i], customList[smallest] = customList[smallest], customList[i]
        heapify(customList, n, smallest)


def heapSort(customList):
    n = len(customList)
    # Build the heap (bottom-up, starting from the last non-leaf node)
    for i in range(int(n / 2) - 1, -1, -1):
        heapify(customList, n, i)

    # Move the root to the end, shrink the heap, re-heapify
    for i in range(n - 1, 0, -1):
        customList[i], customList[0] = customList[0], customList[i]
        heapify(customList, i, 0)
    # customList.reverse()   <- uncomment for ascending order
    return customList


# Example -- min-heap version yields DESCENDING order
nums = [2, 1, 7, 6, 5, 3, 4, 9, 8]
demo("heapSort (desc)", list(nums), heapSort(nums))

heapSort (desc)  [2, 1, 7, 6, 5, 3, 4, 9, 8]  ->  [9, 8, 7, 6, 5, 4, 3, 2, 1]


 Max-heap variant → ascending order
 
 Same algorithm, comparisons flipped from `<` to `>`.

In [12]:
def heapifyMax(customList, n, i):
    largest = i
    l = 2 * i + 1
    r = 2 * i + 2
    if l < n and customList[l] > customList[largest]:
        largest = l
    if r < n and customList[r] > customList[largest]:
        largest = r
    if largest != i:
        customList[i], customList[largest] = customList[largest], customList[i]
        heapifyMax(customList, n, largest)


def heapSortAsc(customList):
    n = len(customList)
    for i in range(n // 2 - 1, -1, -1):
        heapifyMax(customList, n, i)
    for i in range(n - 1, 0, -1):
        customList[i], customList[0] = customList[0], customList[i]
        heapifyMax(customList, i, 0)
    return customList


nums = [2, 1, 7, 6, 5, 3, 4, 9, 8]
demo("heapSort (asc)", list(nums), heapSortAsc(nums))

heapSort (asc)   [2, 1, 7, 6, 5, 3, 4, 9, 8]  ->  [1, 2, 3, 4, 5, 6, 7, 8, 9]


## 8. Sanity check
all algorithms on the same inputEvery implementation should agree with Python's built-in `sorted()`.

In [13]:
data = [2, 1, 7, 6, 5, 3, 4, 9, 8]
expected = sorted(data)

print(f"expected        {expected}\n")

print("bubbleSort      ", bubbleSort(list(data)) == expected)
print("selectionSort   ", selectionSort(list(data)) == expected)
print("insertionSort   ", insertionSort(list(data)) == expected)
print("bucketSort      ", bucketSort(list(data)) == expected)
print("mergeSort       ", mergeSort(list(data), 0, len(data) - 1) == expected)

q = list(data)
quickSort(q, 0, len(q) - 1)
print("quickSort       ", q == expected)

print("heapSortAsc     ", heapSortAsc(list(data)) == expected)
print("heapSort (desc) ", heapSort(list(data)) == expected[::-1])

expected        [1, 2, 3, 4, 5, 6, 7, 8, 9]

bubbleSort       True
selectionSort    True
insertionSort    True
bucketSort       True
mergeSort        True
quickSort        True
heapSortAsc      True
heapSort (desc)  True
